# Emotion Classification - Experiment 2 (Sharvan)

**Tasks 4 & 5 - second training version + push best model to Hugging Face**

This is the second of my two runs. The **only** thing I change from Experiment 1 is the
per-device batch size: 32 -> 16. Smaller batches mean more gradient updates per epoch, which
usually helps a model this size on a small subset, at the cost of a slightly noisier loss curve.
Everything else (lr, epochs, weight decay, seed, subset) is held fixed so the W&B comparison
isolates the effect of batch size.

At the end I push whichever model wins to the Hugging Face Hub and record the link in the W&B
run summary (Task 5).

In [ ]:
!pip install -q transformers datasets scikit-learn wandb

## 1. Load secrets from Kaggle

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()

os.environ["WANDB_PROJECT"] = "mlops-emotion-classification"

## 2. Load the data and the model

Same data preparation as Experiment 1 - identical seed and subset sizes so the only moving
part between the two runs is the batch size.

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

with open("/kaggle/input/id2label/id2label.json") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)
print(f"Loaded {num_labels} labels: {list(id2label.values())}")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

dataset = load_dataset("dair-ai/emotion")


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)


tokenized = dataset.map(tokenize, batched=True)
train_ds = tokenized["train"].shuffle(seed=42).select(range(8000))
eval_ds = tokenized["validation"].shuffle(seed=42).select(range(2000))
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

## 3. Train and track on W&B

**Experiment 2 config:** lr 2e-5, **batch size 16** (the one change), 3 epochs, weight decay 0.01.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

run = wandb.init(
    project="mlops-emotion-classification",
    name="sharvan-exp2-bs16",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "train_samples": len(train_ds),
        "version": "v2",
        "platform": "Kaggle",
    },
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./results_exp2",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="wandb",
    run_name="sharvan-exp2-bs16",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
final_metrics = trainer.evaluate()
print(final_metrics)

## 4. Push the model to Hugging Face (Task 5)

If this run beats Experiment 1, push the weights and tokenizer to my own public HF repo and
store the link in the W&B run summary so the model is traceable from the dashboard. Replace
`<hf-username>` with your handle (repo must be public for the Docker/Actions inference to pull it).

In [ ]:
hf_repo = "<hf-username>/mlops-emotion-distilbert-sharvan"

model.push_to_hub(hf_repo)
tokenizer.push_to_hub(hf_repo)

run.summary["huggingface_model"] = f"https://huggingface.co/{hf_repo}"
run.summary["final_accuracy"] = final_metrics.get("eval_accuracy")
run.summary["final_f1"] = final_metrics.get("eval_f1")
wandb.finish()
print(f"Pushed to https://huggingface.co/{hf_repo}")